# Búsqueda k-NN sobre un corpus · implementación desde cero

k-NN (*k-nearest neighbors*) significa: «dado un punto query, encuentra los **k** puntos más cercanos del corpus». Es la operación base del retrieval.

Aquí lo hacemos con `sentence-transformers` + `numpy`, sin base vectorial. El "motor" completo son tres líneas: **encode → multiplicación → argsort**.

## 1. Importar librerías

- `numpy`: matrices y cálculos (la multiplicación y el ordenado).
- `SentenceTransformer`: el modelo que convierte texto en vectores (embeddings).

Si falta alguna: `uv add numpy sentence-transformers` (nunca `pip`).

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

## 2. Cargar el modelo

`paraphrase-multilingual-MiniLM-L12-v2` es un modelo **multilingüe**: entiende español, inglés y más idiomas, y compara significados, no palabras. Cada frase se convierte en un vector de **384 números**.

La primera vez se descarga de internet (~470 MB); después queda en caché.

In [ ]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

## 3. El corpus

El corpus es nuestra "base de conocimiento": una lista de frases. Cuantas más frases, más interesante la búsqueda; aquí usamos 16 para poder leerlas todas de un vistazo.

Hay frases de capitales, de fútbol y de monumentos: así podremos ver cómo el buscador separa los temas.

In [ ]:
corpus = [
    "Madrid es la capital de España y la ciudad más poblada del país.",
    "El Real Madrid es uno de los clubes de fútbol más importantes.",
    "París es la capital de Francia, situada al norte del país.",
    "El Barcelona ganó la Liga Española en 2023.",
    "Roma es la capital de Italia y centro histórico del Imperio Romano.",
    "Berlín es la capital y ciudad más grande de Alemania.",
    "La torre Eiffel es el monumento más visitado de París.",
    "El Camp Nou es el estadio del FC Barcelona.",
    "El Coliseo de Roma es el anfiteatro más grande que construyó el Imperio Romano.",
    "La Puerta de Brandeburgo es un símbolo de la ciudad de Berlín.",
    "El Atlético de Madrid juega sus partidos en el estadio Metropolitano.",
    "El Sevilla FC ha ganado varias veces la Liga Europa.",
    "La Sagrada Familia es una basílica diseñada por Gaudí en Barcelona.",
    "El Museo del Prado es una de las pinacotecas más importantes de Madrid.",
    "La Juventus es el club de fútbol más laureado de Italia.",
    "El Louvre, en París, es el museo de arte más visitado del mundo.",
]

print(len(corpus), "frases en el corpus")

## 4. Indexación: embeddings de todo el corpus

`model.encode(corpus)` pasa **cada frase** por el modelo y devuelve un vector por frase. Todos juntos forman una matriz de NumPy.

- `normalize_embeddings=True` deja cada vector con **longitud 1**. Así, más adelante, el producto escalar entre dos vectores es directamente su **similitud coseno** (de -1 a 1; más cerca de 1 = más parecido).
- Se hace **una sola vez**. Es lo único "caro"; después buscar es rápido.

`corpus_embeddings.shape` es `(N, 384)`: una fila por frase (N = `len(corpus)`) y 384 números por fila.

In [ ]:
corpus_embeddings = model.encode(corpus, normalize_embeddings=True)

print(corpus_embeddings.shape)  # (número de frases, 384)

## 5. La función `buscar`

Hace 3 pasos, y son el motor completo de un buscador semántico:

1. **encode**: convertir la query en un vector, con el **mismo modelo** y también normalizado.
2. **multiplicación**: `corpus_embeddings @ query_embedding` calcula la similitud de la query con **todas** las frases de golpe.
   - `@` es el operador de NumPy para multiplicar matrices.
   - Matriz `(N, 384)` por vector `(384,)` = `N` números, uno por frase.
3. **argsort**: ordenar y quedarnos con los `k` mejores.
   - `np.argsort(lista)` no devuelve los valores ordenados, sino las **posiciones** (índices) de menor a mayor.
   - Por eso usamos `-similitudes`: al cambiar el signo, "mayor similitud" pasa a ser "menor valor" y quedan primero las más parecidas.
   - `[:k]` se queda solo con las `k` primeras posiciones.

Devuelve una lista de pares `(frase, score)`. El score numérico permite filtrar por umbral más adelante.

In [ ]:
def buscar(query, k=3):
    # 1. Convertimos la query en vector (encode espera una lista, por eso [query]).
    #    [0] = nos quedamos con el único vector que sale.
    query_embedding = model.encode([query], normalize_embeddings=True)[0]

    # 2. Similitud de la query con todo el corpus en una sola multiplicación
    similitudes = corpus_embeddings @ query_embedding  # shape: (N,)

    # 3. Posiciones ordenadas de mayor a menor similitud
    posiciones_mejores = np.argsort(-similitudes)[:k]

    # 4. Con esas posiciones sacamos la frase y su score
    resultados = []
    for posicion in posiciones_mejores:
        frase = corpus[posicion]
        score = float(similitudes[posicion])  # float() convierte el número de NumPy a un float normal
        resultados.append((frase, score))

    return resultados

## 6. Pruebas

Probamos con dos queries. Fíjate en el número que sale delante de cada resultado: es la similitud (más alta = más parecido).

**a) Pregunta directa:** «¿qué ciudad es la capital francesa?». La palabra "París" no aparece en la query, y aun así debería salir primero.

In [ ]:
for frase, score in buscar("¿qué ciudad es la capital francesa?", k=3):
    print(f"{score:.3f} · {frase}")

**b) Asociación por significado:** «equipos de fútbol catalanes». La palabra «catalán» no aparece literalmente en el corpus: el modelo capta la asociación geográfica entre Cataluña y el Barcelona.

In [ ]:
for frase, score in buscar("equipos de fútbol catalanes", k=2):
    print(f"{score:.3f} · {frase}")

## 7. Tres cosas a notar

1. **El "motor" son tres líneas:** `encode`, multiplicación y `argsort`. Una base vectorial no hace conceptualmente más que esto; añade persistencia (guardar en disco), escalabilidad e índices aproximados (no comparar con todos los vectores).
2. **Busca por significado, no por palabras:** «equipos de fútbol catalanes» encuentra el Barcelona aunque «catalán» no aparezca literalmente.
3. **Los resultados llevan score numérico**, para poder filtrar por umbral (por ejemplo, descartar resultados con score < 0.5).